<a href="https://colab.research.google.com/github/Keshav-Agrawal654/Keshav_23FE10CSE00810_MLlab/blob/main/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, roc_auc_score, roc_curve)

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
                                     Conv1D, MaxPooling1D, GlobalAveragePooling1D,
                                     Flatten, Input)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

print("TensorFlow version:", tf.__version__)

# ── Helper & constants ──────────────────────────────────────────
def parse_age(s):
    m = re.search(r'(\d+)\s*years?', str(s))
    return int(m.group(1)) if m else None

mut_cols = ['IDH1','TP53','ATRX','PTEN','EGFR','CIC','MUC16','PIK3CA',
            'NF1','PIK3R1','FUBP1','RB1','NOTCH1','BCOR','CSMD3',
            'SMARCA4','GRIN2A','IDH2','FAT4','PDGFRA']

# ── Load & Preprocess ───────────────────────────────────────────
df = pd.read_csv("TCGA_GBM_LGG_Mutations_all.csv")
df['Age'] = df['Age_at_diagnosis'].apply(parse_age)
for c in mut_cols:
    df[c] = (df[c] == 'MUTATED').astype(int)
df['Gender_num'] = (df['Gender'] == 'Male').astype(int)
df['target'] = (df['Grade'] == 'GBM').astype(int)
df = df.dropna(subset=['Age'])

print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:")
print(df['target'].value_counts())
df.head()

# ── Missing Values ──────────────────────────────────────────────
print("\nMissing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# ── Feature / Target Split ──────────────────────────────────────
feature_cols = mut_cols + ['Age', 'Gender_num']
X = df[feature_cols]
y = df['target']

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target vector shape:  {y.shape}")

# ── Train / Test Split ──────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Testing set size:  {X_test.shape[0]}")
print(f"\nTraining class distribution:\n{y_train.value_counts()}")
print(f"\nTesting class distribution:\n{y_test.value_counts()}")

# ── Feature Scaling ─────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ── Reshape for CNN (samples, timesteps, channels) ──────────────
# CNN expects 3D input: (batch, steps, features)
# We treat each feature as a timestep with 1 channel
X_train_cnn = X_train_scaled.reshape(X_train_scaled.shape[0],
                                      X_train_scaled.shape[1], 1)
X_test_cnn  = X_test_scaled.reshape(X_test_scaled.shape[0],
                                     X_test_scaled.shape[1], 1)

print(f"\nCNN Input shape (train): {X_train_cnn.shape}")
print(f"CNN Input shape (test):  {X_test_cnn.shape}")
print(f"  → (samples, timesteps/features, channels)")

# ── Build CNN Model ─────────────────────────────────────────────
def build_cnn(input_shape):
    model = Sequential([
        # Conv Block 1
        Conv1D(filters=64, kernel_size=3, activation='relu',
               padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        # Conv Block 2
        Conv1D(filters=128, kernel_size=3, activation='relu',
               padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        # Conv Block 3
        Conv1D(filters=64, kernel_size=3, activation='relu',
               padding='same'),
        BatchNormalization(),
        GlobalAveragePooling1D(),
        Dropout(0.3),

        # Fully Connected layers
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),

        Dense(32, activation='relu'),
        Dropout(0.2),

        # Output
        Dense(1, activation='sigmoid')
    ])
    return model

input_shape = (X_train_cnn.shape[1], X_train_cnn.shape[2])
model_cnn = build_cnn(input_shape)
model_cnn.summary()

# ── Compile Model ───────────────────────────────────────────────
model_cnn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# ── Callbacks ───────────────────────────────────────────────────
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=10,
    min_lr=1e-6,
    verbose=1
)

# ── Train Model ─────────────────────────────────────────────────
history = model_cnn.fit(
    X_train_cnn, y_train,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print(f"\nTraining stopped at epoch: {len(history.history['loss'])}")

# ── Training History Plots ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('CNN - Training & Validation Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(history.history['accuracy'], label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('CNN - Training & Validation Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ── Evaluate Model ──────────────────────────────────────────────
y_pred_proba = model_cnn.predict(X_test_cnn).flatten()
y_pred = (y_pred_proba >= 0.5).astype(int)

loss, accuracy = model_cnn.evaluate(X_test_cnn, y_test, verbose=0)
print(f"\nTest Loss:     {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred,
      target_names=['LGG (0)', 'GBM (1)']))

roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")

# ── Confusion Matrix Heatmap ────────────────────────────────────
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['LGG', 'GBM'],
            yticklabels=['LGG', 'GBM'])
plt.title('Confusion Matrix - CNN', fontsize=14, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

print(f"\nTrue Negatives:  {cm[0][0]}")
print(f"False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]}")
print(f"True Positives:  {cm[1][1]}")

# ── ROC Curve ───────────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='purple', lw=2,
         label=f'CNN ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2,
         linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - CNN', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ── Threshold Analysis ──────────────────────────────────────────
from sklearn.metrics import precision_score, recall_score, f1_score
thresholds_to_test = [0.3, 0.4, 0.5, 0.6, 0.7]
print("\nThreshold Analysis:")
print(f"{'Threshold':<12} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1':<12}")
print("-" * 60)
for t in thresholds_to_test:
    y_t = (y_pred_proba >= t).astype(int)
    acc  = accuracy_score(y_test, y_t)
    prec = precision_score(y_test, y_t, zero_division=0)
    rec  = recall_score(y_test, y_t, zero_division=0)
    f1   = f1_score(y_test, y_t, zero_division=0)
    print(f"{t:<12.1f} {acc:<12.4f} {prec:<12.4f} {rec:<12.4f} {f1:<12.4f}")

# ── ANN vs CNN Comparison ───────────────────────────────────────
# Run ANN quickly for comparison
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

ann = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    BatchNormalization(), Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(), Dropout(0.3),
    Dense(32, activation='relu'),
    BatchNormalization(), Dropout(0.2),
    Dense(16, activation='relu'), Dropout(0.2),
    Dense(1, activation='sigmoid')
])
ann.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
ann.fit(X_train_scaled, y_train, epochs=200, batch_size=32,
        validation_split=0.2,
        callbacks=[EarlyStopping(monitor='val_loss', patience=20,
                                 restore_best_weights=True),
                   ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10)],
        verbose=0)

ann_proba = ann.predict(X_train_scaled).flatten()
ann_proba_test = ann.predict(X_test_scaled).flatten()
ann_pred = (ann_proba_test >= 0.5).astype(int)
ann_acc = accuracy_score(y_test, ann_pred)
ann_auc = roc_auc_score(y_test, ann_proba_test)

cnn_acc = accuracy
cnn_auc = roc_auc

comparison = pd.DataFrame({
    'Metric': ['Test Accuracy', 'ROC-AUC Score', 'Total Parameters'],
    'ANN': [f'{ann_acc:.4f}', f'{ann_auc:.4f}', f'{ann.count_params():,}'],
    'CNN': [f'{cnn_acc:.4f}', f'{cnn_auc:.4f}', f'{model_cnn.count_params():,}']
})
print("\n── ANN vs CNN Comparison ──")
print(comparison.to_string(index=False))

# ROC comparison plot
fpr_ann, tpr_ann, _ = roc_curve(y_test, ann_proba_test)
fpr_cnn, tpr_cnn, _ = roc_curve(y_test, y_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr_ann, tpr_ann, color='darkorange', lw=2,
         label=f'ANN (AUC = {ann_auc:.4f})')
plt.plot(fpr_cnn, tpr_cnn, color='purple', lw=2,
         label=f'CNN (AUC = {cnn_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2,
         linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve Comparison - ANN vs CNN', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ── Model Architecture Summary ──────────────────────────────────
print("\nCNN Architecture:")
print(f"  Input shape:     ({X_train_cnn.shape[1]}, 1)")
print(f"  Conv blocks:     3 (64 → 128 → 64 filters)")
print(f"  Pooling:         MaxPooling1D + GlobalAveragePooling1D")
print(f"  Dense layers:    2 (64 → 32)")
print(f"  Output:          1 neuron (Sigmoid)")
print(f"  Total params:    {model_cnn.count_params():,}")
print(f"\nFinal Results:")
print(f"  Test Accuracy:  {accuracy:.4f}")
print(f"  ROC-AUC Score:  {roc_auc:.4f}")

TensorFlow version: 2.19.0
Dataset shape: (857, 30)

Target distribution:
target
0    497
1    360
Name: count, dtype: int64

Missing values per column:
Series([], dtype: int64)

Feature matrix shape: (857, 22)
Target vector shape:  (857,)

Training set size: 685
Testing set size:  172

Training class distribution:
target
0    397
1    288
Name: count, dtype: int64

Testing class distribution:
target
0    100
1     72
Name: count, dtype: int64

CNN Input shape (train): (685, 22, 1)
CNN Input shape (test):  (172, 22, 1)
  → (samples, timesteps/features, channels)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_3 (Conv1D)               │ (None, 22, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 22, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 11, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 11, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 11, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 11, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 5, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 5, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 5, 64)          │        24,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 5, 64)          │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 57,153 (223.25 KB)

 Trainable params: 56,513 (220.75 KB)

 Non-trainable params: 640 (2.50 KB)

Epoch 1/200


KeyboardInterrupt: 